# 04_interpret_final_rf_with_shap

**Role.** Final Random Forest interpretation using model importance and SHAP outputs.

**Pipeline version.** Reproducible scientific pipeline v2 for Dak Lak 2024 coffee mapping Paper 1.


In [1]:
# =============================================================================
# REPRODUCIBILITY BOOTSTRAP: Coffee Paper 1 pipeline v2
# =============================================================================
from pathlib import Path
import os, sys, json, warnings
import numpy as np

# Locate project root robustly whether the notebook is opened from project root
# or from the notebooks/ folder.
_candidate_roots = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _candidate_roots if (p / "config" / "paper1_config.yaml").exists()), Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from coffeemap.config import load_config, ensure_project_dirs, class_info, class_colors, coffee_class_ids
from coffeemap.manifest import init_run_manifest, append_manifest_note
from coffeemap.plotting import set_publication_style

CONFIG = load_config(PROJECT_ROOT / "config" / "paper1_config.yaml")
PATHS = ensure_project_dirs(CONFIG, PROJECT_ROOT)
CLASS_INFO = class_info(CONFIG)
CLASS_COLORS = class_colors(CONFIG)
CLASS_IDS = sorted(CLASS_INFO.keys())
CLASS_NAMES = [CLASS_INFO[i] for i in CLASS_IDS]
COFFEE_CLASSES = coffee_class_ids(CONFIG)
RANDOM_SEED = int(CONFIG.get("project", {}).get("random_seed", 42))
np.random.seed(RANDOM_SEED)

TABLES_DIR = PATHS["tables_dir"]
FIGURES_DIR = PATHS["figures_dir"]
SUPPLEMENTARY_DIR = PATHS["supplementary_dir"]
METADATA_DIR = PATHS["metadata_dir"]
INPUT_DIR = PATHS["input_dir"]

NOTEBOOK_NAME = "03_interpret_final_rf_with_shap.ipynb"
MANIFEST = init_run_manifest(CONFIG, PROJECT_ROOT, notebook_name=NOTEBOOK_NAME)
set_publication_style(font="Arial", dpi=600)

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Classes: {len(CLASS_IDS)} | Coffee classes: {COFFEE_CLASSES} | Random seed: {RANDOM_SEED}")


Project root: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak
Notebook: 03_interpret_final_rf_with_shap.ipynb
Classes: 10 | Coffee classes: [1, 2, 3] | Random seed: 2024


## Reproducibility contract

This notebook follows the project-level configuration in `config/paper1_config.yaml` and writes outputs only under `results/`.

Key safeguards used in this pipeline:

- class IDs, class names, colors, paths, random seed, and coffee class definitions come from one config file;
- each notebook refreshes `results/metadata/run_manifest.json`;
- feature selection must use training data only;
- validation data are reserved for final assessment;
- Olofsson-style estimates are reported as **area-weighted error-adjusted estimates** unless a mapped-class stratified area-assessment sample is available;
- RF uncertainty is interpreted as **RF vote-based class probability**, not calibrated posterior probability.


In [2]:
# =============================================================================
# Pipeline-level imports commonly used by downstream cells
# =============================================================================
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from coffeemap.io import find_file, read_table, write_table
from coffeemap.schema import (
    detect_column, detect_label_columns, assert_class_ids,
    class_count_table, warn_if_balanced, extract_probability_columns,
    assert_probability_matrix,
)
from coffeemap.metrics import classification_summary, overall_metrics, shannon_entropy, probability_margin
from coffeemap.validation import audit_validation_predictions
from coffeemap.olofsson import error_matrix_counts, area_adjustment, binary_coffee_area_adjustment

SEARCH_DIRS = [INPUT_DIR, PATHS["interim_dir"], TABLES_DIR, SUPPLEMENTARY_DIR, PROJECT_ROOT]
print("Reproducible pipeline helpers loaded.")


Reproducible pipeline helpers loaded.


## A. RF + SHAP processing


In [3]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, List, Dict, Tuple
import json
import warnings
import os

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt

try:
    import shap
except ImportError as e:
    raise ImportError(
        "The 'shap' package is required. Install it first, e.g. pip install shap"
    ) from e

print("Libraries loaded.")
print("shap version:", shap.__version__)

Libraries loaded.
shap version: 0.48.0


In [4]:
# =============================================================================
# 1. SETTINGS
# =============================================================================

# Preferred order:
# - data/ is the curated/frozen input folder for Python.
# - GEE_Exports_R3000/ is the raw GEE export folder.
INPUT_DIRS = [
    Path("data/raw"),
    Path("data/raw/data_DakLak_Statistics"),
    Path("data"),
    Path("GEE_Exports_R3000"),
    Path("."),
]

TRAIN_FILE = "Table_TrainSamples_RF_Final_2024.csv"
VAL_FILE   = "Table_ValSamples_RF_Final_2024.csv"
SELECTED_BANDS_FILE = "Table_SelectedBands_corrTop25.csv"
GEE_VAL_PRED_FILE   = "Table_ValPredictions_RF_Final_2024.csv"  # optional audit only

LABEL_COL = "class_id"
OUT_DIR = SUPPLEMENTARY_DIR

# RF parameters aligned with final GEE / manuscript settings.
RANDOM_STATE = 2024
RF_TREES = 2000
RF_MAX_FEATURES = 3
RF_MAX_SAMPLES = 0.65
RF_MIN_SAMPLES_LEAF = 1

# SHAP settings.
# SHAP on all validation samples may be heavy with 2000 trees.
# 600–1000 is usually enough for stable global interpretation.
MAX_SHAP_SAMPLES = 800
SHAP_SAMPLE_SOURCE = "validation"  # "validation" recommended; "train" also supported
RUN_DIAGNOSTIC_SHAP_PLOTS = False
RUN_DEPENDENCE_PLOTS = False

CLASS_NAMES = {
    1: "Sun coffee",
    2: "Intercrop coffee",
    3: "Newly planted coffee",
    4: "Rubber",
    5: "Partially vegetative",
    6: "Rice",
    7: "Other upland crops",
    8: "Forest",
    9: "Water",
    10: "Built",
}
CLASS_IDS = list(CLASS_NAMES.keys())
COFFEE_CLASSES = [1, 2, 3]

print("Output directory:", OUT_DIR.resolve())

Output directory: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary


In [5]:
# =============================================================================
# 2. HELPER FUNCTIONS
# =============================================================================

def find_file(filename: str, required: bool = True) -> Path | None:
    for d in INPUT_DIRS:
        p = d / filename
        if p.exists():
            print(f"Found {filename}: {p}")
            return p
    if required:
        raise FileNotFoundError(
            f"Could not find {filename}. Searched in: {[str(d) for d in INPUT_DIRS]}"
        )
    print(f"Optional file not found: {filename}")
    return None


def read_csv_safely(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, low_memory=False)


def get_selected_features(selected_path: Path | None, train_df: pd.DataFrame, val_df: pd.DataFrame) -> List[str]:
    metadata_cols = {
        LABEL_COL, "classification",
        "system:index", ".geo", "lon", "lat", "longitude", "latitude",
        "split", "gridId", "grid_id", "rnd", "random", "corr_rnd",
    }
    metadata_cols.update(c for c in train_df.columns if c.startswith("Unnamed"))

    if selected_path is not None and selected_path.exists():
        sel = pd.read_csv(selected_path)
        if "band" in sel.columns:
            feats = sel["band"].dropna().astype(str).tolist()
        elif "feature" in sel.columns:
            feats = sel["feature"].dropna().astype(str).tolist()
        elif sel.shape[1] == 1:
            feats = sel.iloc[:, 0].dropna().astype(str).tolist()
        else:
            raise ValueError(
                f"Cannot detect selected-band column in {selected_path}. Columns: {sel.columns.tolist()}"
            )
        missing = [f for f in feats if f not in train_df.columns or f not in val_df.columns]
        if missing:
            raise ValueError(f"Selected features missing from train/val tables: {missing[:10]}")
        return feats

    # Fallback: all numeric non-metadata columns.
    feats = [
        c for c in train_df.columns
        if c in val_df.columns
        and c not in metadata_cols
        and pd.api.types.is_numeric_dtype(train_df[c])
        and pd.api.types.is_numeric_dtype(val_df[c])
    ]
    if not feats:
        raise ValueError("No numeric feature columns detected.")
    warnings.warn(
        f"No selected-band file found. Falling back to all numeric features: {len(feats)} features.",
        RuntimeWarning
    )
    return feats


def clean_xy(df: pd.DataFrame, features: List[str], label_col: str) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    keep = df.copy()
    keep[label_col] = pd.to_numeric(keep[label_col], errors="coerce")
    for f in features:
        keep[f] = pd.to_numeric(keep[f], errors="coerce")

    valid = keep[features].notna().all(axis=1) & keep[label_col].notna()
    dropped = int((~valid).sum())
    if dropped > 0:
        print(f"Dropped {dropped} rows with missing label/features.")

    keep = keep.loc[valid].reset_index(drop=True)
    X = keep[features].astype(float)
    y = keep[label_col].astype(int)
    return X, y, keep


def stratified_sample_indices(y: pd.Series, max_n: int, seed: int = 2024) -> np.ndarray:
    y = pd.Series(y).reset_index(drop=True)
    n = len(y)
    if n <= max_n:
        return np.arange(n)

    rng = np.random.default_rng(seed)
    counts = y.value_counts().sort_index()
    alloc = np.floor(counts / counts.sum() * max_n).astype(int)
    alloc[alloc == 0] = 1

    while alloc.sum() > max_n:
        idx = alloc.idxmax()
        alloc.loc[idx] -= 1
    while alloc.sum() < max_n:
        idx = counts.idxmax()
        alloc.loc[idx] += 1

    sampled = []
    for cls, k in alloc.items():
        idxs = y.index[y == cls].to_numpy()
        k = min(k, len(idxs))
        sampled.extend(rng.choice(idxs, size=k, replace=False).tolist())

    return np.array(sorted(sampled))


def normalize_shap_output(raw_shap, n_samples: int, n_features: int, class_labels: List[int]) -> np.ndarray:
    """
    Convert SHAP output to shape (K, n_samples, n_features).

    Handles common SHAP versions:
    - list of K arrays: each (n, p)
    - ndarray (n, p, K)
    - ndarray (K, n, p)
    - ndarray (n, p) for binary/single-output fallback
    """
    K = len(class_labels)

    if isinstance(raw_shap, list):
        return np.stack(raw_shap, axis=0)

    arr = np.asarray(raw_shap)

    if arr.ndim == 3:
        if arr.shape == (n_samples, n_features, K):
            return np.moveaxis(arr, 2, 0)
        if arr.shape == (K, n_samples, n_features):
            return arr
        if arr.shape[0] == n_samples and arr.shape[1] == n_features:
            return np.moveaxis(arr, 2, 0)
        if arr.shape[1] == n_samples and arr.shape[2] == n_features:
            return arr
        raise ValueError(f"Unexpected 3D SHAP shape: {arr.shape}")

    if arr.ndim == 2:
        return arr[None, :, :]

    raise ValueError(f"Unexpected SHAP output shape: {arr.shape}")


def compute_classwise_table(cm: np.ndarray, class_ids: List[int]) -> pd.DataFrame:
    rows = []
    row_sum = cm.sum(axis=1)
    col_sum = cm.sum(axis=0)
    diag = np.diag(cm)

    for i, cid in enumerate(class_ids):
        pa = diag[i] / row_sum[i] if row_sum[i] > 0 else 0.0  # producer / recall
        ua = diag[i] / col_sum[i] if col_sum[i] > 0 else 0.0  # user / precision
        f1 = 2 * pa * ua / (pa + ua) if (pa + ua) > 0 else 0.0
        rows.append({
            "class_id": cid,
            "class_name": CLASS_NAMES.get(cid, str(cid)),
            "validation_samples": int(row_sum[i]),
            "users_accuracy": ua,
            "producers_accuracy": pa,
            "f1_score": f1,
            "users_accuracy_percent": ua * 100,
            "producers_accuracy_percent": pa * 100,
            "f1_percent": f1 * 100,
        })
    return pd.DataFrame(rows)


def coffee_binary_f1(y_true, y_pred) -> float:
    yt = np.isin(np.asarray(y_true), COFFEE_CLASSES).astype(int)
    yp = np.isin(np.asarray(y_pred), COFFEE_CLASSES).astype(int)
    return f1_score(yt, yp, pos_label=1, zero_division=0)


def coffee_subclass_macro_f1(y_true, y_pred) -> float:
    return f1_score(
        np.asarray(y_true),
        np.asarray(y_pred),
        labels=COFFEE_CLASSES,
        average="macro",
        zero_division=0,
    )


def feature_to_sensor(name: str) -> str:
    if name.startswith("S2_"):
        return "Sentinel-2"
    if name.startswith("S1_"):
        return "Sentinel-1"
    if name.startswith("L89_") or name.startswith("L8_") or name.startswith("L9_"):
        return "Landsat 8/9"
    if name.startswith("DEM_"):
        return "DEM"
    return "Other"

In [6]:
# =============================================================================
# 3. LOAD INPUTS
# =============================================================================

train_path = find_file(TRAIN_FILE, required=True)
val_path = find_file(VAL_FILE, required=True)
selected_path = find_file(SELECTED_BANDS_FILE, required=False)
gee_val_pred_path = find_file(GEE_VAL_PRED_FILE, required=False)

train_raw = read_csv_safely(train_path)
val_raw = read_csv_safely(val_path)

print("Train raw shape:", train_raw.shape)
print("Val raw shape:  ", val_raw.shape)

features = get_selected_features(selected_path, train_raw, val_raw)
print(f"Selected features: {len(features)}")
print(features)

if len(features) != 25:
    warnings.warn(f"Expected 25 selected features, but found {len(features)}.", RuntimeWarning)

X_train, y_train, train_clean = clean_xy(train_raw, features, LABEL_COL)
X_val, y_val, val_clean = clean_xy(val_raw, features, LABEL_COL)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("Train class counts:")
print(y_train.value_counts().sort_index().to_string())
print("Validation class counts:")
print(y_val.value_counts().sort_index().to_string())

pd.DataFrame({"feature": features}).to_csv(OUT_DIR / "selected_features_used_by_RF_SHAP.csv", index=False)

Found Table_TrainSamples_RF_Final_2024.csv: data\raw\Table_TrainSamples_RF_Final_2024.csv
Found Table_ValSamples_RF_Final_2024.csv: data\raw\Table_ValSamples_RF_Final_2024.csv
Found Table_SelectedBands_corrTop25.csv: data\raw\Table_SelectedBands_corrTop25.csv
Found Table_ValPredictions_RF_Final_2024.csv: data\raw\Table_ValPredictions_RF_Final_2024.csv
Train raw shape: (2100, 28)
Val raw shape:   (900, 28)
Selected features: 25
['DEM_elevation', 'S2_dry_CVI', 'S2_dry_MNDWI', 'L89_dry_SWIR1', 'L89_dry_NIR', 'S2_dry_B2', 'S1_VHasc_p50', 'L89_dry_Green', 'S2_wet_B12', 'L89_dry_MSI', 'S2_dry_B5', 'L89_wet_NBR2', 'S2_dry_NDI', 'S2_dry_NDMI', 'S2_wet_NBR2', 'S2_wet_B8', 'L89_wet_MSI', 'S2_wet_NDMI', 'L89_wet_NIR', 'L89_wet_Green', 'L89_wet_NDMI', 'S2_dry_NBR2', 'S2_wet_CVI', 'L89_dry_BSI', 'L89_wet_SWIR2']
X_train: (2100, 25)
X_val:   (900, 25)
Train class counts:
class_id
1     210
2     210
3     210
4     210
5     210
6     210
7     210
8     210
9     210
10    210
Validation class coun

In [7]:
# =============================================================================
# 4. TRAIN PYTHON RF INTERPRETATION MODEL
# =============================================================================

rf = RandomForestClassifier(
    n_estimators=RF_TREES,
    max_features=RF_MAX_FEATURES,
    max_samples=RF_MAX_SAMPLES,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf.fit(X_train, y_train)

class_labels = [int(c) for c in rf.classes_]
print("RF fitted.")
print("RF classes:", class_labels)
print("Number of trees:", len(rf.estimators_))

RF fitted.
RF classes: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Number of trees: 2000


In [8]:
# =============================================================================
# 5. VALIDATION EVALUATION + EXPORTS
# =============================================================================

y_pred = rf.predict(X_val)
proba = rf.predict_proba(X_val)

oa = accuracy_score(y_val, y_pred)
kappa = cohen_kappa_score(y_val, y_pred)
macro_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)
coffee_bin_f1 = coffee_binary_f1(y_val, y_pred)
coffee_sub_f1 = coffee_subclass_macro_f1(y_val, y_pred)

print(f"OA: {oa:.4f}")
print(f"Kappa: {kappa:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Coffee binary F1: {coffee_bin_f1:.4f}")
print(f"Coffee subclass macro F1: {coffee_sub_f1:.4f}")

cm = confusion_matrix(y_val, y_pred, labels=CLASS_IDS)
cm_df = pd.DataFrame(cm, index=CLASS_IDS, columns=CLASS_IDS)
cm_df.to_csv(OUT_DIR / "confusion_matrix_counts.csv", encoding="utf-8-sig")

table4 = compute_classwise_table(cm, CLASS_IDS)
table4.to_csv(OUT_DIR / "Table4_ClassWiseAccuracy_PythonRF_SHAP.csv", index=False, encoding="utf-8-sig")

report = classification_report(
    y_val,
    y_pred,
    labels=CLASS_IDS,
    target_names=[CLASS_NAMES[c] for c in CLASS_IDS],
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(report).T.to_csv(OUT_DIR / "classification_report_python_rf.csv", encoding="utf-8-sig")

val_pred = val_clean.copy()
val_pred["classification_python_rf"] = y_pred
val_pred["correct_python_rf"] = (y_val.to_numpy() == y_pred).astype(int)
for i, cls in enumerate(class_labels):
    val_pred[f"prob_class_{cls}"] = proba[:, i]
val_pred.to_csv(OUT_DIR / "validation_predictions_python_rf.csv", index=False, encoding="utf-8-sig")

metrics = pd.DataFrame([{
    "model": "Python_RF_interpretation_clone",
    "OA": oa,
    "Kappa": kappa,
    "MacroF1": macro_f1,
    "CoffeeBinaryF1": coffee_bin_f1,
    "CoffeeSubclassMacroF1": coffee_sub_f1,
    "n_train": len(y_train),
    "n_val": len(y_val),
    "n_features": len(features),
    "rf_trees": RF_TREES,
    "rf_max_features": RF_MAX_FEATURES,
    "rf_max_samples": RF_MAX_SAMPLES,
    "rf_min_samples_leaf": RF_MIN_SAMPLES_LEAF,
    "random_state": RANDOM_STATE,
}])
metrics.to_csv(OUT_DIR / "rf_shap_model_metrics.csv", index=False, encoding="utf-8-sig")

print("Saved validation metrics, confusion matrix, Table 4 audit, and predictions.")

OA: 0.9356
Kappa: 0.9284
Macro F1: 0.9353
Coffee binary F1: 0.9741
Coffee subclass macro F1: 0.9028
Saved validation metrics, confusion matrix, Table 4 audit, and predictions.


In [9]:
# =============================================================================
# 5B. RF HYPERPARAMETER SENSITIVITY (Supplementary Table S6)
# =============================================================================
# One parameter (n_estimators, mtry, bag fraction) is varied at a time around
# the manuscript configuration (2,000 trees, mtry = 3, bag fraction = 0.65),
# reusing the same 25 selected predictors and the same train/validation split
# loaded above for the SHAP replica. Not comparable to the GEE Smile Random
# Forest results in Tables 3-4 (implementation differs; see Section 2.7).

import itertools

n_estimators_grid = [500, 1000, 1500, 2000, 3000]
max_features_grid = [2, 3, 5]
max_samples_grid = [0.5, 0.65, 0.8, 1.0]
baseline_config = (RF_TREES, RF_MAX_FEATURES, RF_MAX_SAMPLES)

# Sort feature columns alphabetically for this sweep only (leaving the
# importance-ranked X_train/X_val above untouched for SHAP/RF-importance
# cells). With mtry < n_features, sklearn RF draws its per-split feature
# subset by column position, so results are sensitive to column order;
# alphabetical order matches the original Table S6 computation and
# reproduces its numbers exactly.
X_train_sens = X_train[sorted(X_train.columns)]
X_val_sens = X_val[sorted(X_val.columns)]

sensitivity_rows = []
for n_est, mtry, bag in itertools.product(n_estimators_grid, max_features_grid, max_samples_grid):
    sens_clf = RandomForestClassifier(
        n_estimators=n_est,
        max_features=mtry,
        max_samples=bag,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        bootstrap=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    sens_clf.fit(X_train_sens, y_train)
    sens_pred = sens_clf.predict(X_val_sens)
    sensitivity_rows.append({
        "n_estimators": n_est,
        "max_features": mtry,
        "max_samples": bag,
        "OA": accuracy_score(y_val, sens_pred),
        "Kappa": cohen_kappa_score(y_val, sens_pred),
        "MacroF1": f1_score(y_val, sens_pred, average="macro", zero_division=0),
        "CoffeeSubclassMacroF1": coffee_subclass_macro_f1(y_val, sens_pred),
        "is_manuscript_config": (n_est, mtry, bag) == baseline_config,
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_path = OUT_DIR / "RF_hyperparameter_sensitivity_full.csv"
sensitivity_df.to_csv(sensitivity_path, index=False, encoding="utf-8-sig")
print(f"Saved: {sensitivity_path}")

manuscript_row = sensitivity_df[sensitivity_df["is_manuscript_config"]]
print("Manuscript-configuration row (should match Supplementary Table S6's baseline OA/Kappa/MacroF1/CoffeeF1 = 93.33/92.59/93.30/89.86%):")
display(manuscript_row)


Saved: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\RF_hyperparameter_sensitivity_full.csv
Manuscript-configuration row (should match Supplementary Table S6's baseline OA/Kappa/MacroF1/CoffeeF1 = 93.33/92.59/93.30/89.86%):


,n_estimators,max_features,max_samples,OA,Kappa,MacroF1,CoffeeSubclassMacroF1,is_manuscript_config
41,2000,3,0.65,0.933333,0.925926,0.932958,0.89857,True


In [10]:
# =============================================================================
# 6. OPTIONAL AUDIT AGAINST GEE VALIDATION PREDICTIONS
# =============================================================================
# This is only an audit. Python RF and GEE RF are not expected to be identical.

if gee_val_pred_path is not None:
    try:
        gee_pred = read_csv_safely(gee_val_pred_path)
        if "classification" in gee_pred.columns and len(gee_pred) == len(val_pred):
            audit = pd.DataFrame({
                "class_id": y_val.to_numpy(),
                "python_rf": y_pred,
                "gee_rf": pd.to_numeric(gee_pred["classification"], errors="coerce").astype("Int64"),
            })
            audit["python_matches_gee"] = (audit["python_rf"].astype("Int64") == audit["gee_rf"]).astype(int)
            audit.to_csv(OUT_DIR / "python_vs_gee_validation_prediction_audit.csv", index=False, encoding="utf-8-sig")
            print("Python vs GEE prediction agreement:", audit["python_matches_gee"].mean())
        else:
            print("GEE validation prediction file found, but row count or 'classification' column did not match.")
    except Exception as e:
        print("Skipping GEE prediction audit due to error:", e)
else:
    print("No GEE validation prediction file found; skipping audit.")

Python vs GEE prediction agreement: 0.99


In [11]:
# =============================================================================
# 7. RF GINI IMPORTANCE
# =============================================================================

rf_imp = pd.DataFrame({
    "feature": features,
    "rf_importance": rf.feature_importances_,
})
rf_imp["rank"] = rf_imp["rf_importance"].rank(ascending=False, method="first").astype(int)
rf_imp["sensor"] = rf_imp["feature"].apply(feature_to_sensor)
rf_imp = rf_imp.sort_values("rf_importance", ascending=False).reset_index(drop=True)

rf_imp.to_csv(OUT_DIR / "rf_feature_importance.csv", index=False, encoding="utf-8-sig")
print("Top RF features:")
print(rf_imp.head(15).to_string(index=False))

Top RF features:
      feature  rf_importance  rank      sensor
   S2_dry_CVI       0.070743     1  Sentinel-2
 S2_dry_MNDWI       0.070214     2  Sentinel-2
DEM_elevation       0.061366     3         DEM
  S2_wet_NBR2       0.058045     4  Sentinel-2
 S1_VHasc_p50       0.056612     5  Sentinel-1
  S2_dry_NBR2       0.051394     6  Sentinel-2
  L89_dry_NIR       0.046819     7 Landsat 8/9
L89_dry_SWIR1       0.046645     8 Landsat 8/9
    S2_dry_B2       0.046462     9  Sentinel-2
  L89_dry_BSI       0.045924    10 Landsat 8/9
   S2_dry_NDI       0.042083    11  Sentinel-2
L89_dry_Green       0.041279    12 Landsat 8/9
  L89_dry_MSI       0.041084    13 Landsat 8/9
 L89_wet_NBR2       0.038665    14 Landsat 8/9
   S2_wet_B12       0.037893    15  Sentinel-2


In [12]:
# =============================================================================
# 7B. RF GINI IMPORTANCE: 5-SEED STABILITY (adds error bars to Figure 6)
# =============================================================================
# Same hyperparameters and train split as the primary model (cell above), refit
# across 5 seeds to quantify importance stability. Point estimates in rf_imp/
# Figure 6 remain from the primary RANDOM_STATE=2024 fit; this only adds SD.
IMPORTANCE_SEEDS = [2024, 2025, 2026, 2027, 2028]
importance_runs = []
for seed in IMPORTANCE_SEEDS:
    rf_seed = RandomForestClassifier(
        n_estimators=RF_TREES,
        max_features=RF_MAX_FEATURES,
        max_samples=RF_MAX_SAMPLES,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        bootstrap=True,
        random_state=seed,
        n_jobs=-1,
    )
    rf_seed.fit(X_train, y_train)
    importance_runs.append(pd.Series(rf_seed.feature_importances_, index=features, name=seed))

importance_matrix = pd.concat(importance_runs, axis=1)  # rows=features, cols=seeds
rf_imp_5seed = pd.DataFrame({
    "feature": importance_matrix.index,
    "rf_importance_mean": importance_matrix.mean(axis=1).values,
    "rf_importance_sd": importance_matrix.std(axis=1, ddof=1).values,
})
rf_imp_5seed.to_csv(OUT_DIR / "rf_feature_importance_5seed.csv", index=False, encoding="utf-8-sig")
print(f"5-seed RF importance stability computed ({len(IMPORTANCE_SEEDS)} seeds); saved rf_feature_importance_5seed.csv")
print(f"  mean SD across {len(features)} features: {rf_imp_5seed['rf_importance_sd'].mean():.5f}")


5-seed RF importance stability computed (5 seeds); saved rf_feature_importance_5seed.csv
  mean SD across 25 features: 0.00068


In [13]:
# =============================================================================
# 8. SHAP SAMPLE SELECTION
# =============================================================================

if SHAP_SAMPLE_SOURCE.lower() == "train":
    X_source = X_train
    y_source = y_train
else:
    X_source = X_val
    y_source = y_val

shap_idx = stratified_sample_indices(y_source, MAX_SHAP_SAMPLES, seed=RANDOM_STATE)
X_shap = X_source.iloc[shap_idx].reset_index(drop=True)
y_shap_true = y_source.iloc[shap_idx].astype(int).reset_index(drop=True)

print("X_shap:", X_shap.shape)
print("SHAP sample class counts:")
print(y_shap_true.value_counts().sort_index().to_string())

X_shap.to_csv(OUT_DIR / "X_shap_sample.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"class_id": y_shap_true}).to_csv(OUT_DIR / "y_shap_true.csv", index=False, encoding="utf-8-sig")

X_shap: (800, 25)
SHAP sample class counts:
class_id
1     80
2     80
3     80
4     80
5     80
6     80
7     80
8     80
9     80
10    80


In [14]:
# =============================================================================
# 9. COMPUTE SHAP VALUES
# =============================================================================

explainer = shap.TreeExplainer(rf)

print("Computing SHAP values. This may take several minutes...")
raw_shap = explainer.shap_values(X_shap, check_additivity=False)

shap_array = normalize_shap_output(
    raw_shap,
    n_samples=X_shap.shape[0],
    n_features=X_shap.shape[1],
    class_labels=class_labels,
)

print("Normalized SHAP array shape (K, n, p):", shap_array.shape)

# -------------------------------------------------------------------------
# Predicted-class SHAP metadata for Fig. 6A
# -------------------------------------------------------------------------
pred_labels_shap = rf.predict(X_shap)
class_to_index = {int(c): i for i, c in enumerate(class_labels)}
pred_class_indices = np.array([class_to_index[int(c)] for c in pred_labels_shap], dtype=int)

print("Predicted labels in SHAP sample:")
print(pd.Series(pred_labels_shap).value_counts().sort_index().to_string())
print("pred_class_indices shape:", pred_class_indices.shape)

# Signed SHAP values for the class predicted by the RF model.
signed_pred_class_shap = np.vstack([
    shap_array[pred_class_indices[i], i, :]
    for i in range(X_shap.shape[0])
])

print("Predicted-class signed SHAP matrix:", signed_pred_class_shap.shape)

Computing SHAP values. This may take several minutes...


Normalized SHAP array shape (K, n, p): (10, 800, 25)


Predicted labels in SHAP sample:
1     68
2     90
3     81
4     75
5     84
6     73
7     82
8     85
9     79
10    83
pred_class_indices shape: (800,)
Predicted-class signed SHAP matrix: (800, 25)


In [15]:
# =============================================================================
# 10. EXPORT SHAP IMPORTANCE TABLES
# =============================================================================

# Global importance based on predicted-class SHAP values.
# This is the table used by Fig. 6 for global SHAP contribution.
shap_global = pd.DataFrame({
    "feature": features,
    "mean_abs_shap": np.abs(signed_pred_class_shap).mean(axis=0),
    "mean_signed_shap": signed_pred_class_shap.mean(axis=0),
})
shap_global["rank"] = shap_global["mean_abs_shap"].rank(ascending=False, method="first").astype(int)
shap_global["sensor"] = shap_global["feature"].apply(feature_to_sensor)
shap_global = shap_global.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
shap_global.to_csv(OUT_DIR / "shap_global_importance.csv", index=False, encoding="utf-8-sig")

print("Top global SHAP features:")
print(shap_global.head(15).to_string(index=False))

# Class-specific SHAP importance for all model outputs.
for cid in class_labels:
    k = class_to_index[int(cid)]
    d = pd.DataFrame({
        "feature": features,
        "mean_abs_shap": np.abs(shap_array[k]).mean(axis=0),
        "mean_signed_shap": shap_array[k].mean(axis=0),
    })
    d["rank"] = d["mean_abs_shap"].rank(ascending=False, method="first").astype(int)
    d["class_id"] = int(cid)
    d["class_name"] = CLASS_NAMES.get(int(cid), str(cid))
    d["sensor"] = d["feature"].apply(feature_to_sensor)
    d = d.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    d.to_csv(OUT_DIR / f"shap_class_{int(cid)}_importance.csv", index=False, encoding="utf-8-sig")

# Coffee class-specific subset for convenience.
coffee_rows = []
for cid in COFFEE_CLASSES:
    p = OUT_DIR / f"shap_class_{cid}_importance.csv"
    if p.exists():
        d = pd.read_csv(p)
        d["class_id"] = cid
        d["class_name"] = CLASS_NAMES[cid]
        coffee_rows.append(d)
if coffee_rows:
    pd.concat(coffee_rows, ignore_index=True).to_csv(
        OUT_DIR / "shap_coffee_class_importance_long.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("Saved SHAP global and class-specific importance tables.")

Top global SHAP features:
      feature  mean_abs_shap  mean_signed_shap  rank      sensor
   S2_dry_CVI       0.063775          0.062791     1  Sentinel-2
 S2_dry_MNDWI       0.056267          0.054843     2  Sentinel-2
  S2_dry_NBR2       0.050026          0.049219     3  Sentinel-2
  S2_wet_NBR2       0.049583          0.048277     4  Sentinel-2
  L89_dry_BSI       0.041266          0.040734     5 Landsat 8/9
    S2_dry_B2       0.040485          0.038613     6  Sentinel-2
DEM_elevation       0.039811          0.034646     7         DEM
 S1_VHasc_p50       0.038981          0.035911     8  Sentinel-1
  L89_dry_NIR       0.034928          0.033846     9 Landsat 8/9
L89_dry_Green       0.031162          0.030361    10 Landsat 8/9
  L89_dry_MSI       0.030663          0.029847    11 Landsat 8/9
L89_dry_SWIR1       0.029879          0.028492    12 Landsat 8/9
   S2_dry_NDI       0.029745          0.029249    13  Sentinel-2
 L89_wet_NBR2       0.029033          0.027406    14 Landsat 8/9

In [16]:
# =============================================================================
# 11. SAVE SHAP NPZ FOR FIGURE 6
# =============================================================================

np.savez_compressed(
    OUT_DIR / "shap_values_all.npz",
    shap_values=shap_array,                         # (K, n, p)
    signed_pred_class_shap=signed_pred_class_shap,  # (n, p)
    X_shap=X_shap.values,
    feature_names=np.array(features, dtype=object),
    class_labels=np.array(class_labels, dtype=int),
    pred_class_indices=pred_class_indices,
    pred_labels_shap=np.array(pred_labels_shap, dtype=int),
    y_shap_true=np.array(y_shap_true, dtype=int),
)

print("Saved:", OUT_DIR / "shap_values_all.npz")
print("This NPZ now contains pred_class_indices and pred_labels_shap for Fig. 6A.")

Saved: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\shap_values_all.npz
This NPZ now contains pred_class_indices and pred_labels_shap for Fig. 6A.


In [17]:
# =============================================================================
# 12. OPTIONAL DIAGNOSTIC SHAP PLOTS
# =============================================================================
# Final Fig. 6 is generated by 06_fig6_SHAP_with_RF_comparison.py.
# The plots below are optional diagnostic/supporting outputs.

if RUN_DIAGNOSTIC_SHAP_PLOTS:
    plt.figure()
    shap.summary_plot(
        signed_pred_class_shap,
        X_shap,
        feature_names=features,
        show=False,
        max_display=20,
    )
    plt.tight_layout()
    plt.savefig(OUT_DIR / "diagnostic_predicted_class_shap_summary.png", dpi=600, bbox_inches="tight")
    plt.close()

    for cid in COFFEE_CLASSES:
        if cid not in class_to_index:
            continue
        k = class_to_index[cid]
        plt.figure()
        shap.summary_plot(
            shap_array[k],
            X_shap,
            feature_names=features,
            show=False,
            max_display=20,
        )
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"diagnostic_shap_summary_class_{cid}.png", dpi=600, bbox_inches="tight")
        plt.close()

    print("Saved diagnostic SHAP summary plots.")
else:
    print("Diagnostic SHAP plots skipped.")

Diagnostic SHAP plots skipped.


In [18]:
# =============================================================================
# 13. OPTIONAL DEPENDENCE PLOTS
# =============================================================================
# Only run if you want supplementary diagnostic plots. Not used in final Fig. 6.

if RUN_DEPENDENCE_PLOTS:
    top_features = shap_global["feature"].head(5).tolist()
    for cid in COFFEE_CLASSES:
        if cid not in class_to_index:
            continue
        k = class_to_index[cid]
        for feat in top_features:
            try:
                plt.figure()
                shap.dependence_plot(
                    feat,
                    shap_array[k],
                    X_shap,
                    interaction_index="auto",
                    show=False,
                )
                plt.title(f"Dependence: {feat} | class {cid}: {CLASS_NAMES[cid]}")
                plt.tight_layout()
                safe_feat = feat.replace("/", "_").replace(" ", "_")
                plt.savefig(OUT_DIR / f"diagnostic_dependence_{safe_feat}_class_{cid}.png",
                            dpi=600, bbox_inches="tight")
                plt.close()
            except Exception as e:
                print(f"Dependence plot failed for {feat}, class {cid}: {e}")
    print("Dependence plots complete.")
else:
    print("Dependence plots skipped.")

Dependence plots skipped.


In [19]:
# =============================================================================
# 14. RUN MANIFEST
# =============================================================================

manifest = {
    "purpose": "Python RF interpretation clone for SHAP and RF importance",
    "train_file": str(train_path),
    "val_file": str(val_path),
    "selected_bands_file": str(selected_path) if selected_path else None,
    "n_train": int(len(y_train)),
    "n_val": int(len(y_val)),
    "n_features": int(len(features)),
    "features": features,
    "class_labels": class_labels,
    "rf_parameters": {
        "n_estimators": RF_TREES,
        "max_features": RF_MAX_FEATURES,
        "max_samples": RF_MAX_SAMPLES,
        "min_samples_leaf": RF_MIN_SAMPLES_LEAF,
        "bootstrap": True,
        "random_state": RANDOM_STATE,
    },
    "validation_metrics": {
        "OA": float(oa),
        "Kappa": float(kappa),
        "MacroF1": float(macro_f1),
        "CoffeeBinaryF1": float(coffee_bin_f1),
        "CoffeeSubclassMacroF1": float(coffee_sub_f1),
    },
    "shap": {
        "sample_source": SHAP_SAMPLE_SOURCE,
        "max_shap_samples": MAX_SHAP_SAMPLES,
        "actual_shap_samples": int(X_shap.shape[0]),
        "shap_array_shape_K_n_p": list(shap_array.shape),
        "predicted_class_indices_saved": True,
    },
    "note": (
        "This Python RF is used for interpretation and reproducibility auditing. "
        "The final mapped product is generated by the GEE RF model."
    ),
}

with open(OUT_DIR / "rf_shap_run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

with open(OUT_DIR / "rf_shap_run_manifest.txt", "w", encoding="utf-8") as f:
    f.write("RF-SHAP run manifest\\n")
    f.write("====================\\n")
    f.write(f"Train file: {train_path}\\n")
    f.write(f"Validation file: {val_path}\\n")
    f.write(f"Selected bands file: {selected_path}\\n")
    f.write(f"n_train: {len(y_train)}\\n")
    f.write(f"n_val: {len(y_val)}\\n")
    f.write(f"n_features: {len(features)}\\n")
    f.write(f"OA: {oa:.4f}\\n")
    f.write(f"Kappa: {kappa:.4f}\\n")
    f.write(f"Macro F1: {macro_f1:.4f}\\n")
    f.write(f"Coffee binary F1: {coffee_bin_f1:.4f}\\n")
    f.write(f"Coffee subclass macro F1: {coffee_sub_f1:.4f}\\n")
    f.write(f"SHAP samples: {X_shap.shape[0]}\\n")
    f.write("pred_class_indices saved in shap_values_all.npz: yes\\n")

print("Workflow complete.")
print("Outputs saved to:", OUT_DIR.resolve())
print("Key files:")
for name in [
    "rf_feature_importance.csv",
    "shap_global_importance.csv",
    "shap_values_all.npz",
    "shap_class_1_importance.csv",
    "shap_class_2_importance.csv",
    "shap_class_3_importance.csv",
    "validation_predictions_python_rf.csv",
    "confusion_matrix_counts.csv",
    "rf_shap_run_manifest.json",
]:
    print(" -", OUT_DIR / name)

Workflow complete.
Outputs saved to: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary
Key files:
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\rf_feature_importance.csv
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\shap_global_importance.csv
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\shap_values_all.npz
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\shap_class_1_importance.csv
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\shap_class_2_importance.csv
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\shap_class_3_importance.csv
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\validation_predictions_python_rf.csv
 - D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\confusion_matr

## B. Figure 5: RF variable importance panels


In [20]:
"""
05, Figure 5: Variable importance in 3 panels
=============================================================================
Panel A: top 25 predictors ranked by RF Gini importance, colored by sensor
Panel B: relative contribution of each sensor (S2/S1/L89/DEM) as % of total
Panel C: aggregated importance by functional group

Input:
    SUPPLEMENTARY_DIR/rf_feature_importance.csv   (written by cell 76bc2b17 above)

Output:
    FIGURES_DIR/Figure6_VariableImportance.png
    FIGURES_DIR/Figure6_VariableImportance.pdf
    SUPPLEMENTARY_DIR/fig5_functional_group_mapping.csv
=============================================================================
"""

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

IN_CSV = SUPPLEMENTARY_DIR / "rf_feature_importance.csv"
TOP_N = 25


# -----------------------------------------------------------------------------
# 1) SENSOR CLASSIFICATION: by prefix
# -----------------------------------------------------------------------------
SENSOR_COLORS = {
    'Sentinel-2':  '#2e7d32',
    'Sentinel-1':  '#1565c0',
    'Landsat 8/9': '#ef6c00',
    'DEM':         '#6a1b9a',
}

def feature_to_sensor(name: str) -> str:
    if name.startswith('S2_'):  return 'Sentinel-2'
    if name.startswith('S1_'):  return 'Sentinel-1'
    if name.startswith('L89_'): return 'Landsat 8/9'
    if name.startswith('DEM_'): return 'DEM'
    return 'Other'


# -----------------------------------------------------------------------------
# 2) FUNCTIONAL-GROUP CLASSIFICATION
# -----------------------------------------------------------------------------
GROUP_COLORS = {
    'Moisture':          '#00838f',
    'Vegetation':        '#2e7d32',
    'Red-edge':          '#CC79A7',
    'Burn / soil':       '#bf360c',
    'Radar backscatter': '#1565c0',
    'Radar texture':     '#4527a0',
    'Topography':        '#6a1b9a',
    'Raw spectral':      '#757575',
}

FUNCTIONAL_PATTERNS = [
    (r'(^|_)(NDMI|NDWI|MNDWI|MSI)(_|$)',                    'Moisture'),
    (r'(^|_)(NBR|NBR2|BSI|SWIR2_p75)(_|$)',                 'Burn / soil'),
    (r'(^|_)(NDVI_RE|CI_RE|MTCI|PSRI|B5|B6|B7)(_|$)',       'Red-edge'),
    (r'(^|_)(NDVI|GNDVI|RDVI|NLI|MSR|EVI|CVI|NDI)(_|$)',    'Vegetation'),
    (r'(^|_)GLCM',                                           'Radar texture'),
    (r'glcm',                                                'Radar texture'),
    (r'VH_glcm|VV_glcm',                                     'Radar texture'),
    (r'VVVHratio|VVminusVH|(^|_)(VV|VH)(desc|asc)?(_p\d+|_std|_p50)?(_|$)',
                                                             'Radar backscatter'),
    (r'(^|_)(elevation|slope|aspect|hillshade)(_|$)',        'Topography'),
    (r'(^|_)(B2|B3|B4|B8|B11|B12|Blue|Green|Red|NIR|SWIR1|SWIR2)(_|$)',
                                                             'Raw spectral'),
]


def feature_to_group(name: str) -> str:
    for pat, grp in FUNCTIONAL_PATTERNS:
        if re.search(pat, name):
            return grp
    return 'Raw spectral'


# -----------------------------------------------------------------------------
# 3) LOAD
# -----------------------------------------------------------------------------
df = pd.read_csv(IN_CSV)
if 'feature' not in df.columns:
    for cand in ['variable', 'band', 'predictor']:
        if cand in df.columns:
            df = df.rename(columns={cand: 'feature'})
            break
if 'rf_importance' not in df.columns:
    for cand in ['importance', 'gini_importance', 'MeanDecreaseGini']:
        if cand in df.columns:
            df = df.rename(columns={cand: 'rf_importance'})
            break
if not {'feature', 'rf_importance'}.issubset(df.columns):
    raise ValueError(f"Input must contain feature/rf_importance columns. Found: {df.columns.tolist()}")
print(f'Loaded {len(df)} features from {IN_CSV}')

df['sensor'] = df['feature'].apply(feature_to_sensor)
df['group']  = df['feature'].apply(feature_to_group)
df = df.sort_values('rf_importance', ascending=False).reset_index(drop=True)

_sd_path = SUPPLEMENTARY_DIR / 'rf_feature_importance_5seed.csv'
if _sd_path.exists():
    df = df.merge(pd.read_csv(_sd_path)[['feature', 'rf_importance_sd']], on='feature', how='left')
else:
    df['rf_importance_sd'] = np.nan

df[['feature', 'sensor', 'group', 'rf_importance']].to_csv(
    SUPPLEMENTARY_DIR / 'fig5_functional_group_mapping.csv', index=False
)


# -----------------------------------------------------------------------------
# 4) FIGURE LAYOUT
# -----------------------------------------------------------------------------
fig = plt.figure(figsize=(10, 12))
gs  = GridSpec(2, 3, figure=fig,
               width_ratios=[1.6, 1.0, 1.4],
               height_ratios=[1.0, 1.0],
               wspace=0.45, hspace=0.35)

axA = fig.add_subplot(gs[:, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 1:])
axL = fig.add_subplot(gs[0, 2])
axL.axis('off')

# --- Panel A ---
topA    = df.head(TOP_N).iloc[::-1]
colorsA = [SENSOR_COLORS.get(s, '#999') for s in topA['sensor']]
axA.barh(range(len(topA)), topA['rf_importance'].values, color=colorsA, edgecolor='none',
         xerr=topA['rf_importance_sd'].values, error_kw=dict(elinewidth=0.7, capsize=2, ecolor='#333333'))
axA.set_yticks(range(len(topA)))
axA.set_yticklabels(topA['feature'].values, fontsize=9)
axA.set_xlabel('RF Gini importance', fontsize=10)
axA.text(-0.12, 1.05, 'a', transform=axA.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')
axA.spines['top'].set_visible(False)
axA.spines['right'].set_visible(False)
axA.tick_params(axis='x', labelsize=9)

# --- Panel B ---
sensor_share      = df.groupby('sensor')['rf_importance'].sum()
sensor_share      = sensor_share.reindex([s for s in SENSOR_COLORS if s in sensor_share.index]).dropna()
sensor_pct        = sensor_share / sensor_share.sum() * 100
sensor_pct_plot   = sensor_pct.sort_values(ascending=True)
colorsB           = [SENSOR_COLORS[s] for s in sensor_pct_plot.index]
yB = np.arange(len(sensor_pct_plot))
axB.barh(yB, sensor_pct_plot.values, color=colorsB, edgecolor='none', height=0.62)
axB.set_yticks(yB)
axB.set_yticklabels(sensor_pct_plot.index, fontsize=9)
axB.set_xlim(0, max(sensor_pct_plot.values) * 1.22)
axB.set_xlabel('Share of total RF importance (%)', fontsize=10)
axB.text(-0.12, 1.05, 'b', transform=axB.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')
axB.spines['top'].set_visible(False)
axB.spines['right'].set_visible(False)
axB.spines['left'].set_visible(False)
axB.tick_params(axis='x', labelsize=9)
axB.tick_params(axis='y', length=0)
axB.grid(axis='x', linestyle=':', linewidth=0.6, alpha=0.35)
for i, v in enumerate(sensor_pct_plot.values):
    axB.text(v + max(sensor_pct_plot.values) * 0.025, i, f'{v:.1f}%',
             va='center', ha='left', fontsize=9, color='#333')
axL.axis('off')

# --- Panel C ---
group_share = df.groupby('group')['rf_importance'].sum().sort_values()
colorsC     = [GROUP_COLORS.get(g, '#777') for g in group_share.index]
axC.barh(range(len(group_share)), group_share.values, color=colorsC, edgecolor='none')
axC.set_yticks(range(len(group_share)))
axC.set_yticklabels(group_share.index, fontsize=9)
axC.set_xlabel('Total RF Gini importance', fontsize=10)
axC.text(-0.12, 1.05, 'c', transform=axC.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')
axC.spines['top'].set_visible(False)
axC.spines['right'].set_visible(False)
axC.tick_params(axis='x', labelsize=9)
total = group_share.sum()
for i, v in enumerate(group_share.values):
    axC.text(v + total * 0.01, i, f'{v / total * 100:.1f}%', va='center', fontsize=9, color='#444')

# -----------------------------------------------------------------------------
# SAVE
# -----------------------------------------------------------------------------
png = FIGURES_DIR / 'Figure6_VariableImportance.png'
pdf = FIGURES_DIR / 'Figure6_VariableImportance.pdf'
plt.savefig(png, dpi=600, bbox_inches='tight')
plt.savefig(pdf, bbox_inches='tight')
print(f'Saved: {png}')
print(f'Saved: {pdf}')
print(f'Saved: {SUPPLEMENTARY_DIR / "fig5_functional_group_mapping.csv"}')

plt.close(fig)


Loaded 25 features from D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\rf_feature_importance.csv


Saved: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\figures\Figure6_VariableImportance.png
Saved: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\figures\Figure6_VariableImportance.pdf
Saved: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\fig5_functional_group_mapping.csv


## C. Figure 6: SHAP interpretation and RF–SHAP consistency


In [21]:
"""
06, Figure 6: SHAP interpretation with RF–SHAP consistency check
=============================================================================
Purpose
-------
Final publication-style Figure 6 for Paper 1.
This version keeps the figure scientifically consistent:
    - SHAP panels use mean absolute SHAP values.
    - RF is used only in one explicit comparison panel to test consistency
      between RF-Gini importance and SHAP importance.

Panels
------
A. Global SHAP summary for the predicted class
B. RF vs SHAP importance comparison
C. SHAP contribution by sensor
D. SHAP contribution by functional group
E. Class-specific SHAP importance for the three coffee systems

Inputs expected in: Supplementary/
    rf_feature_importance.csv
    shap_global_importance.csv
    shap_class_1_importance.csv
    shap_class_2_importance.csv
    shap_class_3_importance.csv
    shap_values_all.npz

Outputs in: Figures/
    Figure7_SHAP_Interpretation.png/pdf/svg
    separate panel files
    contribution/merged CSV audit tables
=============================================================================
"""

import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from scipy.stats import spearmanr

# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------
IN_DIR = SUPPLEMENTARY_DIR
OUT_DIR = FIGURES_DIR

# Set SHOW_FIGURES=1 to open interactive windows after saving outputs.
SHOW_FIGURES = os.environ.get("SHOW_FIGURES", "0") == "1"

# -----------------------------------------------------------------------------
# Style
# -----------------------------------------------------------------------------
plt.rcParams.update(
    {
        "font.family": "Arial",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.linewidth": 0.8,
    }
)

SENSOR_COLORS = {
    "Sentinel-2": "#2e7d32",
    "Sentinel-1": "#1565c0",
    "Landsat 8/9": "#ef6c00",
    "DEM": "#6a1b9a",
    "Other": "#999999",
}

GROUP_COLORS = {
    "Raw spectral": "#757575",
    "Vegetation": "#2e7d32",
    "Red-edge": "#CC79A7",
    "Burn / soil": "#bf360c",
    "Moisture": "#00838f",
    "Radar backscatter": "#1565c0",
    "Radar texture": "#4527a0",
    "Topography": "#6a1b9a",
}

COFFEE_COLORS = {1: "#8c3b00", 2: "#d85a30", 3: "#f5a623"}
CLASS_NAMES = {1: "Sun coffee", 2: "Intercrop coffee", 3: "Newly planted coffee"}


# -----------------------------------------------------------------------------
# Mapping rules
# -----------------------------------------------------------------------------
def feature_to_sensor(name: str) -> str:
    if name.startswith("S2_"):
        return "Sentinel-2"
    if name.startswith("S1_"):
        return "Sentinel-1"
    if name.startswith("L89_") or name.startswith("L8_") or name.startswith("L9_"):
        return "Landsat 8/9"
    if name.startswith("DEM_"):
        return "DEM"
    return "Other"


FUNCTIONAL_PATTERNS = [
    (r"(^|_)(NDMI|NDWI|MNDWI|MSI)(_|$)", "Moisture"),
    (r"(^|_)(NBR|NBR2|BSI|SWIR2|SWIR2_p75)(_|$)", "Burn / soil"),
    (r"(^|_)(NDVI_RE|CI_RE|MTCI|PSRI|B5|B6|B7|B8A)(_|$)", "Red-edge"),
    (r"(^|_)(NDVI|GNDVI|RDVI|NLI|MSR|EVI|CVI|NDI)(_|$)", "Vegetation"),
    (r"(^|_)GLCM|glcm|VH_glcm|VV_glcm", "Radar texture"),
    (
        r"VVVHratio|VVminusVH|(^|_)(VV|VH)(desc|asc)?(_p\d+|_std|_p50)?(_|$)",
        "Radar backscatter",
    ),
    (r"(^|_)(elevation|slope|aspect|hillshade|TRI)(_|$)", "Topography"),
    (
        r"(^|_)(B1|B2|B3|B4|B8|B11|B12|Blue|Green|Red|NIR|SWIR1|SWIR2)(_|$)",
        "Raw spectral",
    ),
]


def feature_to_group(name: str) -> str:
    for pat, grp in FUNCTIONAL_PATTERNS:
        if re.search(pat, name):
            return grp
    return "Raw spectral"


# -----------------------------------------------------------------------------
# Load inputs
# -----------------------------------------------------------------------------
rf_imp = pd.read_csv(IN_DIR / "rf_feature_importance.csv")
if "rf_importance" not in rf_imp.columns:
    candidates = [
        c for c in rf_imp.columns if "importance" in c.lower() or "gini" in c.lower()
    ]
    if not candidates:
        raise ValueError(
            "rf_feature_importance.csv must contain 'rf_importance' or an importance-like column."
        )
    rf_imp = rf_imp.rename(columns={candidates[0]: "rf_importance"})

shap_imp = pd.read_csv(IN_DIR / "shap_global_importance.csv")
if "mean_abs_shap" not in shap_imp.columns:
    candidates = [c for c in shap_imp.columns if "shap" in c.lower()]
    if not candidates:
        raise ValueError(
            "shap_global_importance.csv must contain 'mean_abs_shap' or a SHAP-like column."
        )
    shap_imp = shap_imp.rename(columns={candidates[0]: "mean_abs_shap"})

npz = np.load(IN_DIR / "shap_values_all.npz", allow_pickle=True)
shap_values = npz["shap_values"]
X_shap = npz["X_shap"]
feature_names = [str(x) for x in list(npz["feature_names"])]

# Predicted-class signed SHAP for beeswarm.
# Preferred input is written by 02_RF_SHAP_optimized.ipynb.
if "signed_pred_class_shap" in npz:
    signed_sv = npz["signed_pred_class_shap"]
elif shap_values.ndim == 3:
    if "pred_class_indices" in npz:
        pred_idx = npz["pred_class_indices"].astype(int)
    else:
        # Backward-compatible fallback for old SHAP files.
        pred_idx = np.argmax(np.abs(shap_values).sum(axis=2), axis=0)
    signed_sv = np.vstack(
        [shap_values[pred_idx[i], i, :] for i in range(shap_values.shape[1])]
    )
else:
    signed_sv = shap_values

if signed_sv.shape[1] != len(feature_names):
    raise ValueError(
        f"SHAP feature dimension mismatch: signed_sv has {signed_sv.shape[1]} columns, "
        f"but feature_names has {len(feature_names)}."
    )

# Align SHAP global importance.
shap_imp = shap_imp[shap_imp["feature"].isin(feature_names)].copy()
shap_imp["sensor"] = shap_imp["feature"].apply(feature_to_sensor)
shap_imp["group"] = shap_imp["feature"].apply(feature_to_group)
shap_imp = shap_imp.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
shap_imp[["feature", "sensor", "group", "mean_abs_shap"]].to_csv(
    OUT_DIR / "fig6_shap_feature_mapping.csv", index=False
)

# SHAP-based contribution by sensor/group.
sensor_share = shap_imp.groupby("sensor", as_index=False)["mean_abs_shap"].sum()
sensor_share["percent"] = (
    100 * sensor_share["mean_abs_shap"] / sensor_share["mean_abs_shap"].sum()
)
sensor_order = ["Sentinel-2", "Landsat 8/9", "Sentinel-1", "DEM", "Other"]
sensor_share["order"] = sensor_share["sensor"].apply(
    lambda s: sensor_order.index(s) if s in sensor_order else 99
)
sensor_share = sensor_share.sort_values(
    ["order", "percent"], ascending=[True, False]
).drop(columns="order")
sensor_share.to_csv(OUT_DIR / "fig6_shap_sensor_contribution.csv", index=False)

group_share = shap_imp.groupby("group", as_index=False)["mean_abs_shap"].sum()
group_share["percent"] = (
    100 * group_share["mean_abs_shap"] / group_share["mean_abs_shap"].sum()
)
group_share = group_share.sort_values("percent", ascending=True)
group_share.to_csv(OUT_DIR / "fig6_shap_functional_contribution.csv", index=False)

# RF vs SHAP merged table.
merged = (
    rf_imp[["feature", "rf_importance"]]
    .merge(shap_imp[["feature", "mean_abs_shap"]], on="feature", how="inner")
    .dropna()
)
rho, pval = spearmanr(merged["rf_importance"], merged["mean_abs_shap"])
merged.to_csv(OUT_DIR / "fig6_rf_vs_shap_importance.csv", index=False)

# Class-specific SHAP importance.
COFFEE = [1, 2, 3]
cls_imp = {}
for cid in COFFEE:
    p = IN_DIR / f"shap_class_{cid}_importance.csv"
    if p.exists():
        d = pd.read_csv(p)
        if "mean_abs_shap" not in d.columns:
            candidates = [c for c in d.columns if "shap" in c.lower()]
            if candidates:
                d = d.rename(columns={candidates[0]: "mean_abs_shap"})
        cls_imp[cid] = d

if cls_imp:
    union = None
    for cid, d in cls_imp.items():
        top = d.head(12)[["feature", "mean_abs_shap"]].copy()
        top.rename(columns={"mean_abs_shap": f"class_{cid}"}, inplace=True)
        union = top if union is None else union.merge(top, on="feature", how="outer")
    union = union.fillna(0)
    class_cols = [c for c in union.columns if c.startswith("class_")]
    union["combined"] = union[class_cols].sum(axis=1)
    union = union.sort_values("combined", ascending=True).tail(12)
else:
    union = pd.DataFrame(
        columns=["feature", "class_1", "class_2", "class_3", "combined"]
    )
    print(
        "Warning: no shap_class_<id>_importance.csv found; panel E will be shown as unavailable."
    )


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def clean_axes(ax, grid_axis="x"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color="#d9d9d9", linewidth=0.5, alpha=0.65)
        ax.set_axisbelow(True)


def save_figure(fig, name: str):
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(
            OUT_DIR / f"{name}.{ext}",
            dpi=600 if ext == "png" else None,
            bbox_inches="tight",
        )


# -----------------------------------------------------------------------------
# Panels
# -----------------------------------------------------------------------------
def panel_A(ax, fig=None, top_n=20, add_colorbar=True):
    top_features = shap_imp.head(top_n)["feature"].tolist()
    idxs = [feature_names.index(f) for f in top_features][::-1]
    last_sc = None
    for plot_y, feat_idx in enumerate(idxs):
        sv = signed_sv[:, feat_idx]
        xval = X_shap[:, feat_idx]
        if np.nanmax(xval) > np.nanmin(xval):
            xnorm = (xval - np.nanmin(xval)) / (
                np.nanmax(xval) - np.nanmin(xval) + 1e-12
            )
        else:
            xnorm = np.zeros_like(xval)
        rng = np.random.default_rng(2026 + plot_y)
        y_jit = plot_y + rng.uniform(-0.28, 0.28, size=len(sv))
        last_sc = ax.scatter(
            sv,
            y_jit,
            c=xnorm,
            cmap="coolwarm",
            s=4.5,
            alpha=0.55,
            linewidths=0,
            rasterized=True,
            vmin=0,
            vmax=1,
        )
    ax.set_yticks(range(len(idxs)))
    ax.set_yticklabels([feature_names[i] for i in idxs], fontsize=9)
    ax.axvline(0, color="#666666", lw=0.9)
    ax.set_xlabel("SHAP value for predicted class")
    ax.text(-0.12, 1.05, 'a', transform=ax.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#e6e6e6", linewidth=0.45, alpha=0.7)
    if add_colorbar and fig is not None and last_sc is not None:
        cb = fig.colorbar(last_sc, ax=ax, fraction=0.026, pad=0.017)
        cb.ax.set_title("Feature\nvalue", fontsize=10, pad=6)
        cb.set_ticks([0, 1])
        cb.set_ticklabels(["Low", "High"])
        cb.ax.tick_params(labelsize=12)


def panel_B_rf_vs_shap(ax):
    ax.scatter(
        merged["rf_importance"],
        merged["mean_abs_shap"],
        s=18,
        color="#555555",
        alpha=0.75,
        edgecolors="none",
    )
    # Linear visual guide only, not used for inference.
    if len(merged) > 2:
        x = merged["rf_importance"].values
        y = merged["mean_abs_shap"].values
        coef = np.polyfit(x, y, 1)
        xx = np.linspace(np.nanmin(x), np.nanmax(x), 100)
        ax.plot(xx, coef[0] * xx + coef[1], color="#9e9e9e", lw=1.1, zorder=0)

    # Label top features by SHAP; use alternating offsets to minimize overlap.
    top = (
        merged.sort_values("mean_abs_shap", ascending=False)
        .head(5)
        .reset_index(drop=True)
    )
    offsets = [(7, 4), (8, -10), (-44, 7), (7, -12), (-43, -10)]
    for i, r in top.iterrows():
        dx, dy = offsets[i % len(offsets)]
        ax.annotate(
            r["feature"],
            (r["rf_importance"], r["mean_abs_shap"]),
            xytext=(dx, dy),
            textcoords="offset points",
            fontsize=9,
            color="#b71c1c",
            arrowprops=dict(arrowstyle="-", color="#bdbdbd", lw=0.6),
            bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.75),
        )
    ax.text(
        0.03,
        0.94,
        f"Spearman ρ = {rho:.3f}\n({'p < 0.001' if pval < 0.001 else f'p = {pval:.3f}'})",
        transform=ax.transAxes,
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#cccccc", lw=0.5),
    )
    ax.set_xlabel("RF Gini importance")
    ax.set_ylabel("Mean |SHAP|")
    ax.text(-0.12, 1.05, 'b', transform=ax.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')
    clean_axes(ax, grid_axis="both")


def panel_C_sensor(ax):
    d = sensor_share.sort_values("percent", ascending=True).copy()
    colors = [SENSOR_COLORS.get(s, "#999999") for s in d["sensor"]]
    ax.barh(d["sensor"], d["percent"], color=colors, edgecolor="none")
    xmax = max(5, d["percent"].max() * 1.28)
    ax.set_xlim(0, xmax)
    for y, v in enumerate(d["percent"]):
        ax.text(
            v + xmax * 0.018, y, f"{v:.1f}%", va="center", fontsize=9, color="#333333"
        )
    ax.set_xlabel("Share of total mean |SHAP| (%)")
    ax.text(-0.12, 1.05, 'c', transform=ax.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')
    clean_axes(ax, grid_axis="x")


def panel_D_group(ax):
    d = group_share.copy()
    colors = [GROUP_COLORS.get(g, "#777777") for g in d["group"]]
    ax.barh(d["group"], d["percent"], color=colors, edgecolor="none")
    xmax = max(5, d["percent"].max() * 1.25)
    ax.set_xlim(0, xmax)
    for y, v in enumerate(d["percent"]):
        ax.text(
            v + xmax * 0.016, y, f"{v:.1f}%", va="center", fontsize=9, color="#333333"
        )
    ax.set_xlabel("Share of total mean |SHAP| (%)")
    ax.text(-0.12, 1.05, 'd', transform=ax.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')
    clean_axes(ax, grid_axis="x")


def panel_E_class(ax):
    if union.empty:
        ax.text(
            0.5,
            0.5,
            "Class-specific SHAP files not found",
            ha="center",
            va="center",
            fontsize=10,
            color="#555555",
        )
        ax.text(-0.12, 1.05, 'e', transform=ax.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        return

    y_pos = np.arange(len(union))
    bar_h = 0.23
    offsets = {1: -bar_h, 2: 0, 3: bar_h}
    for cid in COFFEE:
        col = f"class_{cid}"
        if col not in union.columns:
            continue
        ax.barh(
            y_pos + offsets[cid],
            union[col].values,
            height=bar_h,
            color=COFFEE_COLORS[cid],
            label=f"{cid}. {CLASS_NAMES[cid]}",
            edgecolor="none",
        )
    ax.set_yticks(y_pos)
    ax.set_yticklabels(union["feature"].values, fontsize=9)
    ax.set_xlabel("Mean |SHAP| within class")
    ax.text(-0.12, 1.05, 'e', transform=ax.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')
    ax.legend(loc="lower right", frameon=False)
    clean_axes(ax, grid_axis="x")


# -----------------------------------------------------------------------------
# Composite figure
# -----------------------------------------------------------------------------
fig = plt.figure(figsize=(10, 13), constrained_layout=False)
gs = GridSpec(
    3,
    3,
    figure=fig,
    height_ratios=[1.13, 0.92, 1.25],
    width_ratios=[1.05, 0.90, 1.60],
    wspace=0.90,
    hspace=0.30,
)

axA = fig.add_subplot(gs[0, 0:2])
axB = fig.add_subplot(gs[0, 2])
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1:3])
axE = fig.add_subplot(gs[2, :])

panel_A(axA, fig=fig, top_n=20, add_colorbar=True)
panel_B_rf_vs_shap(axB)
panel_C_sensor(axC)
panel_D_group(axD)
panel_E_class(axE)

fig.subplots_adjust(bottom=0.085, top=0.97)
save_figure(fig, "Figure7_SHAP_Interpretation")
print(f"Saved: {OUT_DIR / 'Figure7_SHAP_Interpretation.png'}")
print(f"Spearman rho = {rho:.4f}, p = {pval:.2e}")
if SHOW_FIGURES:
    plt.show(block=False)
else:
    plt.close(fig)


Saved: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\figures\Figure7_SHAP_Interpretation.png
Spearman rho = 0.9608, p = 2.64e-14


In [22]:
# =============================================================================
# OUTPUT MANIFEST
# =============================================================================
manifest_rows = []
for root in [TABLES_DIR, FIGURES_DIR, SUPPLEMENTARY_DIR]:
    if root.exists():
        for p in sorted(root.rglob("*")):
            if p.is_file():
                manifest_rows.append({
                    "folder": root.name,
                    "file": str(p.relative_to(root)),
                    "size_kb": round(p.stat().st_size / 1024, 1),
                })
manifest = pd.DataFrame(manifest_rows)
manifest_path = SUPPLEMENTARY_DIR / f"Manifest_{Path().resolve().name}.csv"
manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(manifest.tail(30))
print("Manifest saved:", manifest_path)


,folder,file,size_kb
177,supplementary,S06_block_summary_20km.csv,3.3
178,supplementary,selected_features_by_fold.csv,23.4
179,supplementary,selected_features_used_by_RF_SHAP.csv,0.3
180,supplementary,selection_frequency.csv,9.3
181,supplementary,shap_class_10_importance.csv,2.0
182,supplementary,shap_class_1_importance.csv,2.1
183,supplementary,shap_class_2_importance.csv,2.3
184,supplementary,shap_class_3_importance.csv,2.3
185,supplementary,shap_class_4_importance.csv,2.0
186,supplementary,shap_class_5_importance.csv,2.3


Manifest saved: D:\2024_PhD_Research\Chap2_Mapping\mmlab-coffeemap-daklak\results\supplementary\Manifest_mmlab-coffeemap-daklak.csv
